# Empirical Mode Decomposition (EMD)

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

EMD decomposes the signal into Intrinsic Mode Functions (IMFs) ranging from high to low frequencies. We use the PyEMD library.

## Expected outputs

- Original signal on top
- Five IMFs ranging from high to low frequencies
- First IMF captures high frequencies (artifacts)

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Samples | 5000 | First 25 seconds |
| max_imf | 5 | Max IMFs |


## 1. Install dependencies


In [ ]:
!pip install mne scikit-learn EMD-signal scipy numpy plotly wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply EMD

We apply `EMD` from the PyEMD library to the first 5000 samples of channel P4.


In [ ]:
from PyEMD import EMD

n_plot = min(5000, len(eeg_data))
channel_data = eeg_data[:n_plot, 0]

emd = EMD()
imfs = emd(channel_data, max_imf=5)
print(f'Number of IMFs: {imfs.shape[0]}')


## 5. Interactive plot

**What to look for:**

- First IMF contains the highest frequencies
- Later IMFs contain slower frequencies
- First IMF can be removed to eliminate high-frequency artifacts



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(n_plot) / fs
n_imfs = min(imfs.shape[0], 5)

fig = make_subplots(rows=n_imfs + 1, cols=1, shared_xaxes=True,
                    subplot_titles=['Original'] + [f'IMF {i+1}' for i in range(n_imfs)])
fig.add_trace(go.Scatter(x=t_sec, y=channel_data, name='Original',
                         line=dict(color='black', width=0.5)), row=1, col=1)
for i in range(n_imfs):
    fig.add_trace(go.Scatter(x=t_sec, y=imfs[i], name=f'IMF {i+1}',
                             line=dict(color='blue', width=0.5)), row=i+2, col=1)
fig.update_layout(height=900, title_text='EMD Decomposition - Channel P4',
                  xaxis_title='Time (s)', showlegend=False)
fig.show()


## What did we learn?

- EMD decomposes the signal into IMFs without assuming basis functions
- Early IMFs capture high frequencies, later ones capture low frequencies
- Specific IMFs can be removed to eliminate artifacts
- Suitable for non-stationary signals like EEG

